# RAG Financial Document Analyst

Questions about NVIDIA, Microsoft, JPMorgan, Amazon and Tesla, answered from their latest **10-K filings**
on SEC EDGAR, with every claim cited and checked.

1. **Corpus:** each company's latest 10-K, Item 1A (Risk Factors) and Item 7 (MD&A), split into ~1,200-character chunks,
   plus reported revenue and net income from SEC's XBRL company facts.
2. **Retrieval:** dense embeddings (BGE-small + FAISS), keyword search (BM25), and a hybrid of both, measured on 20 labelled questions.
3. **Answers:** an LLM answers from the retrieved passages; the notebook checks that citations point at real sources
   and that every number in the answer appears in a cited source.

Retrieval and its evaluation run without an LLM, so they're re-run by CI on every push.

In [1]:
# Cell 1 — Setup
import json, os, sys
from datetime import date
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))
PROJECT_DIR = Path.cwd().parent if Path.cwd().name in ("notebooks", "src") else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / "src"))
import edgar, rag, answer as an

CORPUS_DIR = PROJECT_DIR / "corpus"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
# Cell 2 — Corpus: build it from SEC EDGAR once, then reuse the saved copy
version_file = CORPUS_DIR / "version.txt"
if version_file.exists() and version_file.read_text().strip() == edgar.CORPUS_VERSION:
    chunks = pd.read_csv(CORPUS_DIR / "chunks.csv")
    facts = pd.read_csv(CORPUS_DIR / "facts.csv")
    coverage = pd.read_csv(CORPUS_DIR / "coverage.csv")
    print(f"Loaded saved corpus: {len(chunks)} chunks")
else:
    chunks, facts, coverage = edgar.build_corpus(CORPUS_DIR, PROJECT_DIR / "data" / "raw")
    print(f"Built corpus (version {edgar.CORPUS_VERSION}) from SEC EDGAR: {len(chunks)} chunks")
coverage

Loaded saved corpus: 695 chunks


,company,section,found,characters,chunks,accession,period
0,NVIDIA,Risk Factors,True,114825,128,0001045810-26-000021,2026-01-25
1,NVIDIA,MD&A,True,34151,35,0001045810-26-000021,2026-01-25
2,Microsoft,Risk Factors,True,81159,91,0001193125-26-323660,2026-06-30
3,Microsoft,MD&A,True,51214,50,0001193125-26-323660,2026-06-30
4,JPMorgan,Risk Factors,True,112127,115,0001628280-26-008131,2025-12-31
5,JPMorgan,MD&A,False,0,0,0001628280-26-008131,2025-12-31
6,Amazon,Risk Factors,True,60685,69,0001018724-26-000004,2025-12-31
7,Amazon,MD&A,True,46337,50,0001018724-26-000004,2025-12-31
8,Tesla,Risk Factors,True,83732,97,0001628280-26-003952,2025-12-31
9,Tesla,MD&A,True,55447,60,0001628280-26-003952,2025-12-31


In [3]:
# Cell 3 — Reported figures (SEC XBRL), and an audit of the first version's hand-written corpus
shown = facts.assign(revenue=facts["revenue"].map(an.billions), net_income=facts["net_income"].map(an.billions))
display(shown[[c for c in ["company", "fiscal_year_end", "revenue", "net_income", "revenue_concept", "check"] if c in shown]])

# The first version of this project answered from 13 hand-written passages. Their revenue claims, against the filings:
v1_claims_billion = {"NVIDIA": 130.5, "Microsoft": 261.8, "JPMorgan": 175.4, "Amazon": 637.9, "Tesla": 113.4}
audit = facts[["company", "fiscal_year_end", "revenue"]].assign(
    v1_claim=facts["company"].map(v1_claims_billion),
    reported=(facts["revenue"] / 1e9).round(1))
audit["difference_pct"] = ((audit["v1_claim"] - audit["reported"]) / audit["reported"] * 100).round(1)
audit.drop(columns="revenue")

,company,fiscal_year_end,revenue,net_income,revenue_concept
0,NVIDIA,2026-01-25,$215.9 billion,$120.1 billion,Revenues
1,Microsoft,2026-06-30,$331.8 billion,$133.7 billion,RevenueFromContractWithCustomerExcludingAssess...
2,JPMorgan,2025-12-31,$182.4 billion,$57.0 billion,Revenues
3,Amazon,2025-12-31,$716.9 billion,$77.7 billion,RevenueFromContractWithCustomerExcludingAssess...
4,Tesla,2025-12-31,$94.8 billion,$3.8 billion,Revenues


,company,fiscal_year_end,v1_claim,reported,difference_pct
0,NVIDIA,2026-01-25,130.5,215.9,-39.6
1,Microsoft,2026-06-30,261.8,331.8,-21.1
2,JPMorgan,2025-12-31,175.4,182.4,-3.8
3,Amazon,2025-12-31,637.9,716.9,-11.0
4,Tesla,2025-12-31,113.4,94.8,19.6


In [4]:
# Cell 4 — Retrieval evaluation: dense vs keyword vs hybrid, on 20 labelled questions
questions = pd.read_csv(PROJECT_DIR / "eval" / "questions.csv")
retriever = rag.Retriever(chunks)
scores = rag.evaluate(retriever, questions, k=5)
scores.to_csv(OUTPUT_DIR / "retrieval_eval.csv", index=False)
scores

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,mode,hit@5,keyword_hit@5,MRR
0,dense,1.00,0.90,0.960
1,bm25,0.85,0.70,0.760
2,hybrid,0.95,0.95,0.862
3,hybrid+filter,0.95,0.95,0.925


In [5]:
# Cell 5 — What retrieval returns for one question
hits = retriever.search("What risks does NVIDIA describe from export controls?", k=3)
for r in hits.itertuples():
    print(f"[{r.rank}] {r.chunk_id}  {r.text[:220]}...\n")

[1] NVIDIA-RF-098  Export controls have and are likely in the future to have a disproportionate impact on NVIDIA and may disadvantage us against certain of our competitors that sell chips that are outside the scope of such control. Export ...

[2] NVIDIA-RF-083  Export controls and other restrictions targeting GPUs and semiconductors associated with AI, which have been imposed and are likely to be more restrictive, would further limit our ability to export our technology, produc...

[3] NVIDIA-RF-097  negatively impact sales of such products to markets outside China, including the U.S. and Europe. In addition, as the performance of the gaming GPUs increases over time, export controls may have a greater impact on our a...



In [6]:
# Cell 6 — Answers with citations from an LLM (skipped when no LLM endpoint is set, e.g. in CI)
if not os.getenv("OPENAI_BASE_URL") and os.getenv("OPENAI_API_KEY", "lm-studio") in ("", "lm-studio"):
    print("No LLM endpoint set: skipping answers. Set OPENAI_BASE_URL (LM Studio) in the repo's .env to run this cell.")
else:
    chat = an.openai_chat()
    asks = [
        "Which of these companies had the highest revenue in its latest fiscal year?",
        "What risks does NVIDIA describe from export controls?",
        "How did AWS segment sales change at Amazon?",
        "How many vehicles did Tesla deliver, and what does Tesla say drove automotive revenue?",
    ]
    runs = [an.ask(q, retriever, facts, chat, k=8) for q in asks]
    print("Verdicts:", pd.Series([r["verdict"] for r in runs]).value_counts().to_dict(), "\n")
    for r in runs:
        flag = {
            "grounded": "grounded: every citation and number traces to a source",
            "partial": "partial: the model says the retrieved passages don't fully answer this",
            "abstained": "abstained: the model found no answer in the retrieved passages",
            "uncited": "uncited: an answer with no citations",
        }.get(r["verdict"], f"CHECK: unknown citations {r['unknown_citations']}, numbers not in sources {r['numbers_not_in_sources']}")
        print(f"Q: {r['question']}\nA: {r['answer']}\n[{flag}]\n" + "-" * 70)
    json.dump({"model": os.getenv("LLM_MODEL", "gpt-4o-mini"), "date": date.today().isoformat(), "runs": runs},
              open(OUTPUT_DIR / "rag_run.json", "w"), indent=2)

Verdicts: {'grounded': 3, 'partial': 1} 

Q: Which of these companies had the highest revenue in its latest fiscal year?
A: Based on the provided sources, Amazon had the highest revenue in its latest fiscal year, reporting $716.9 billion for the fiscal year ended 2025-12-31 [F].
[grounded: every citation and number traces to a source]
----------------------------------------------------------------------
Q: What risks does NVIDIA describe from export controls?
A: Based on the provided risk factors, NVIDIA outlines the following risks associated with export controls:

**Competitive & Market Position Risks**
- Export controls may disproportionately disadvantage NVIDIA against competitors selling chips outside the control scope [S1].
- Customers outside impacted regions may "design-out" U.S. semiconductors to reduce compliance burdens and ensure global market access [S1].
- Overseas governments may request that customers purchase from competitors rather than NVIDIA or other U.S. firms [S1